# 1 - Yields, shapes and datacards

Reads a campaign's coffea files and writes everything Combine needs.

**MC only.** `shape_tools` refuses the data merge outright, and the fail-closed
blinding guard in `datacard_tools` withholds region A for anything not positively
identified as simulation.

Two histogram layouts are handled by the same module:

| layout | campaigns | extractor |
|---|---|---|
| pre-binned `abcd_region` axis | `cosmic_veto_v1`, `golden_hotspot_iso025_v1`, `muIso_dPhi_eHEM_skimv1_v1` | `st.collect_shapes`, `st.mljlj_two_bin` |
| 6D inclusive, regions sliced here | `sixd_inclusive_v1` | `st.collect_6d` |

The command-line equivalent of this notebook is

```bash
python sidm/scripts/abcd_datacards.py extract --campaign sixd_inclusive_v1 \
    --plane iso_iso,iso_dphi --datacards
```

In [ ]:
import sys
from pathlib import Path

STUDY = Path.cwd()
REPO = STUDY.parents[2]
for p in (str(REPO), str(STUDY)):
    if p not in sys.path:
        sys.path.insert(0, p)

import datacard_tools as dt
import shape_tools as st
import plot_tools as pt

CAMPAIGN = "sixd_inclusive_v1"      # see dt.CAMPAIGNS for the others
dt.use_campaign(CAMPAIGN)
OUT = dt.campaign_outdir(STUDY, CAMPAIGN)
PLOTS = STUDY / "plots" / CAMPAIGN
print(CAMPAIGN, "->", OUT)
print("channels:", {k: v.selection for k, v in dt.CHANNELS.items()})

## 0 - The blinding guard, demonstrated

Run this first. It is the executable form of the guarantee, not documentation of it.

A sample's signal region is released only if the sample can be **positively identified as
simulation** - the `is_data` flag where it carries a usable value, otherwise an allow-list
(a known signal point, or an entry in `cross_sections.yaml`). Everything else is withheld:
missing metadata, a new data era, an unrecognised name, a typo.

The failure mode is *withholding too much*, which is visible. It is never leaking, which
would not be. Below, a real data file returns its control regions and refuses region A.

In [ ]:
# 1. The guard on the active campaign: data is not simulation, and the
#    signal region refuses to be read.
probe = sorted(Path(dt.DATA_DIR).glob("*.coffea"))[0]
channel = dt.CHANNELS["SR_4mu"]
for sample, sample_out in dt.read_coffea(probe).items():
    print(f"{sample}: is_simulation={dt.is_simulation(sample, sample_out)}")
    try:
        dt.sr_yield(sample_out, channel, sample_name=sample)
        print("  !! the signal region was readable -- the guard is NOT working")
    except dt.BlindingError as err:
        print(f"  BlindingError: {str(err)[:110]}...")

# 2. shape_tools refuses the data directory outright, so not even the
#    control regions are reachable from the shape side of the study.
try:
    st.collect_6d(dt.DATA_DIR, "iso_dphi")
    print("!! the data merge was readable -- the guard is NOT working")
except dt.BlindingError as err:
    print(f"collect_6d: {err}")

# 3. Control regions ARE returned -- they are what the background estimate is
#    built from. Shown on a pre-binned campaign, since this one has no
#    abcd_region axis for region_yields to read (it would return None for a
#    missing histogram, which is not the same thing as blinding).
dt.use_campaign("golden_hotspot_iso025_v1")
probe = sorted(Path(dt.DATA_DIR).glob("*.coffea"))[0]
for sample, sample_out in dt.read_coffea(probe).items():
    regions = dt.region_yields(sample_out, dt.CHANNELS["SR_4mu"], sample_name=sample)
    print(f"{sample}: regions returned = "
          f"{[dt.ABCD_REGIONS[r] for r in sorted(regions)] if regions else None}")
dt.use_campaign(CAMPAIGN)           # back to the campaign under study

## Choose the region boundaries

`Selection` holds every cut that defines the regions, so a looser working point
is a keyword rather than an edit. The defaults reproduce the analysis selection.

Loosening isolation alone buys very little - the displacement requirements are
what cost the statistics, so `apply_disp=False` is the move that reaches
hundreds of background events.

In [ ]:
NOMINAL = st.Selection("nominal")                       # A = 27.6 / 3.3
LOOSE   = st.Selection("iso<0.5, no disp",              # A = 3711 / 719
                       iso_cut=0.5, apply_disp=False)

for s in (NOMINAL, LOOSE):
    print(f"{s.name:24s} {s.describe()}")

## Extract the per-region shapes

`collect_6d` returns `{sample: {channel: {region: Shape}}}`; `group_shapes`
sums the samples into the datacard process groups. One plane at a time - the
region A it produces is identical between planes, which is what makes the
plane comparison controlled.

In [ ]:
PLANE = "iso_dphi"          # or "iso_iso"; see st.PLANES

bkg_shapes = st.collect_6d(dt.BKG_DIR, PLANE, selection=NOMINAL)
sig_shapes = st.collect_6d(dt.SIGNAL_DIR, PLANE, selection=NOMINAL)
bkg = st.group_shapes(bkg_shapes)

print("groups:", sorted(bkg), "| signal points:", len(sig_shapes))
for ch in sorted(dt.CHANNELS):
    y = {r: st.total_background_shape(bkg, ch, r).total for r in range(4)}
    a, b, c, d = (y[r].value for r in range(4))
    print(f"{ch:10s} A {a:9.4f}  B {b:9.4f}  C {c:9.4f}  D {d:9.4f}"
          f"  | B*C/D {b*c/d:9.4f}")

## Write the shapes JSON

This is the handover to the Combine release, which has ROOT but not coffea.
It carries the binned distributions, the campaign, and the normalisation.

In [ ]:
observable = st.Observable(name=f"ljlj_mass_{PLANE}", hists={}, axis="ljlj_mass",
                           label=r"$m_{LJLJ}$ [GeV]",
                           note=f"6D slice, plane {PLANE}, {NOMINAL.describe()}")
shapes_json = st.export_shapes(
    OUT / f"shapes_{PLANE}.json", bkg, sig_shapes, observable,
    meta={"normalisation": "signal normalised to 1 fb at 59.83 /fb",
          "plane": PLANE, "selection": NOMINAL.describe()})
print(shapes_json)

## Counting and ABCD datacards

Both are built from the *integral* of the same shapes the parametric cards use,
so the methods differ only in what the datacard does with the bins - not in the
events behind them.

A signal point with no cross section is refused rather than silently normalised
to the 1 fb reference, so the theory-normalised set covers only the points that
have one.

In [ ]:
from sidm.tools import utilities

xs = utilities.load_yaml(f"{dt.BASE_DIR}/configs/cross_sections.yaml")
bkg_yields = st.grouped_yields(bkg)
signal = st.integrated(sig_shapes)
signal_theory = {k: v for k, v in signal.items() if k in xs}
print(f"theory-normalisable: {len(signal_theory)} of {len(signal)}")

for tag, writer, points, cfg in (
        ("counting", dt.write_datacards, signal, dt.DatacardConfig()),
        ("abcd", dt.write_abcd_datacards, signal, dt.DatacardConfig()),
        ("counting_theory", dt.write_datacards, signal_theory,
         dt.DatacardConfig(use_theory_xs=True)),
        ("abcd_theory", dt.write_abcd_datacards, signal_theory,
         dt.DatacardConfig(use_theory_xs=True))):
    written, warn = writer(points, bkg_yields, OUT / f"datacards_{tag}_{PLANE}", config=cfg)
    print(f"{tag:16s} {len(written):4d} cards" + (f", {len(warn)} warnings" if warn else ""))

## Then, outside this notebook

The workspace step needs ROOT, so it runs in the Combine release:

```bash
python sidm/scripts/abcd_workspace.py \
    --shapes  <campaign>/shapes_iso_dphi.json \
    --outdir  <campaign>/datacards_shape_iso_dphi
```

and the limits themselves:

```bash
python sidm/scripts/run_combine_limits.py -j 8 \
    --datacards <campaign>/datacards_shape_iso_dphi \
    --pattern  'datacard_shape_*.txt' \
    --outdir   <campaign>/limits_shape_iso_dphi
```

Notebook 2 picks up from the `limits.csv` those produce.